# Google Colab Pro+ Execution Setup

This notebook walks you through running the mechanistic interpretability code on a Colab Pro+ GPU runtime.

## Setup Instructions:
1. **Select Accelerator**: Click on the connection dropdown in the upper right corner (or go to **Runtime** > **Change runtime type**).
   - **Hardware accelerator**: Select **GPU**.
   - **GPU class**: Choose **Premium** (to request an A100 or V100 GPU) or **Standard** (for a T4 or L4 GPU, which is also extremely fast for this model and consumes fewer compute units).
   - **Runtime shape**: Choose **High-RAM**.
2. **Timeout & Background Execution**: Because you have Colab Pro+, you can enable **Background execution** (in the runtime settings) which allows the notebook to keep running for up to 24 hours even if you close your browser tab!

In [1]:
# Step 1: Mount Google Drive
# (Recommended: Upload project.zip to your Google Drive to easily access it across sessions)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
# Step 2: Unzip Project Files and Change Directory
import os

# Set to True if you uploaded the project.zip directly to the Colab files sidebar (instead of Google Drive):
uploaded_directly_to_colab = False

if uploaded_directly_to_colab:
    zip_path = "/content/project.zip"
else:
    # Path to where you uploaded project.zip in your Google Drive (e.g., in root 'My Drive' folder):
    zip_path = "/content/drive/MyDrive/mphil-project/project.zip"

if os.path.exists(zip_path):
    print(f"Extracting {zip_path}...")
    !unzip -q -o {zip_path} -d /content/
    os.chdir('/content')
    print(f"Current working directory: {os.getcwd()}")
    !ls -l
else:
    print(f"ERROR: Could not find {zip_path}.")
    print("Please upload 'project.zip' to Google Drive or the Colab files sidebar and check the path.")

Extracting /content/drive/MyDrive/mphil-project/project.zip...
Current working directory: /content
total 120
drwxr-xr-x 2 root root  4096 Jul  6 05:16 configs
drwxr-xr-x 2 root root  4096 Jul  6 05:16 data
drwx------ 5 root root  4096 Jul  6 05:16 drive
-rw-r--r-- 1 root root   264 Jul  4 22:38 environment.yml
-rw-r--r-- 1 root root 19186 Jul  4 21:53 IMPLEMENTATION_PLAN.md
-rw-r--r-- 1 root root  1579 Jun  2 01:05 Instructions.md
drwxr-xr-x 3 root root  4096 Jul  6 05:16 mechanistic_data
drwxr-xr-x 2 root root  4096 Jul  6 03:59 mechanistic_interpretability_repro.egg-info
drwxr-xr-x 3 root root  4096 Jul  6 05:16 outputs
-rw-r--r-- 1 root root  6551 Jul  2 18:59 project.txt
drwxr-xr-x 2 root root  4096 Jul  5 19:49 __pycache__
-rw-r--r-- 1 root root  7647 Jul  6 02:51 README.md
-rw-r--r-- 1 root root 36388 Jul  6 04:28 run.ipynb
drwxr-xr-x 1 root root  4096 Jun  4 13:39 sample_data
-rw-r--r-- 1 root root   826 Jul  5 20:12 setup.py
drwxr-xr-x 4 root root  4096 Jul  6 05:16 src


In [3]:
!cp /content/drive/MyDrive/mphil-project/intervention.py /content/src/intervention.py
!ls -l /content/src

total 112
-rw-r--r-- 1 root root 25509 Jul  6 03:54 attribution_graph.py
-rw-r--r-- 1 root root 12847 Jul  5 19:48 baseline.py
-rw-r--r-- 1 root root  7592 Jul  5 20:37 capture_activations.py
-rw-r--r-- 1 root root   648 Jul  5 19:48 config_utils.py
-rw-r--r-- 1 root root  3299 Jul  5 20:13 data_utils.py
-rw-r--r-- 1 root root    76 Jul  5 19:48 __init__.py
-rw-r--r-- 1 root root 18330 Jul  6 05:17 intervention.py
-rw-r--r-- 1 root root  2617 Jul  6 03:38 model_loader.py
drwxr-xr-x 2 root root  4096 Jul  6 05:16 prompts
-rw-r--r-- 1 root root   133 Jul  5 19:48 prompt_utils.py
drwxr-xr-x 2 root root  4096 Jul  6 03:59 __pycache__
-rw-r--r-- 1 root root 10065 Jul  6 00:45 train.py


In [4]:
# Step 3: Install Required Dependencies
# We upgrade transformers to >= 4.51.0 to support the Qwen3 model architecture
!pip install --upgrade "transformers>=4.51.0" accelerate
!pip install -e .

Obtaining file:///content
  Preparing metadata (setup.py) ... done
  Attempting uninstall: mechanistic-interpretability-repro
    Found existing installation: mechanistic-interpretability-repro 0.1.0
    Uninstalling mechanistic-interpretability-repro-0.1.0:
      Successfully uninstalled mechanistic-interpretability-repro-0.1.0
  Running setup.py develop for mechanistic-interpretability-repro


In [5]:
!python src/attribution_graph.py \
  --prompt "Fact: The capital of the country containing Kandahar is named" \
  --target "Kabul" \
  --output-json outputs/kabul_graph.json \
  --output-html outputs/kabul_graph.html \
  --output-mermaid outputs/kabul_graph.md

Loading model and tokenizer...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 179.94it/s]
Disabling gradients for all model parameters for speed and memory efficiency.
Loading SAE models from /content/mechanistic_data/sae_checkpoints to device: cuda...
Loaded SAE for layer 8 (scaling factor: 0.3032)
Loaded SAE for layer 16 (scaling factor: 0.3014)
Loaded SAE for layer 24 (scaling factor: 0.6629)
Loaded SAE for layer 32 (scaling factor: 2.0936)
Running forward pass...

Top 5 model predictions:
  ' Kabul' (prob: 0.9922, logit: 21.50)
  ' K' (prob: 0.0030, logit: 15.69)
  ' as' (prob: 0.0012, logit: 14.81)
  ' after' (prob: 0.0009, logit: 14.50)
  ' Her' (prob: 0.0006, logit: 14.06)

Target token for attribution: ' Kabul' (ID: 86545, probability: 0.9922)
Running backward pass on logit: 21.5000
Computing SAE feature attributions...
Tracing paths and computing edge attribution

In [6]:
# Optional: Verify output files exist and inspect them
!ls -lh outputs/

total 440K
drwxr-xr-x 2 root root 4.0K Jul  6 05:16 baselines
-rw-r--r-- 1 root root  30K Jul  6 04:27 dallas_graph.html
-rw-r--r-- 1 root root  23K Jul  6 04:25 dallas_graph.json
-rw-r--r-- 1 root root 7.6K Jul  6 04:25 dallas_graph.md
-rw-r--r-- 1 root root  90K Jul  6 05:31 georgia_graph.html
-rw-r--r-- 1 root root  62K Jul  6 05:31 georgia_graph.json
-rw-r--r-- 1 root root  26K Jul  6 05:31 georgia_graph.md
-rw-r--r-- 1 root root 2.6K Jul  6 05:06 intervention_results.json
-rw-r--r-- 1 root root  90K Jul  6 05:37 kabul_graph.html
-rw-r--r-- 1 root root  62K Jul  6 05:37 kabul_graph.json
-rw-r--r-- 1 root root  26K Jul  6 05:37 kabul_graph.md


In [7]:
!mkdir -p /content/drive/MyDrive/mphil-project/outputs
!cp /content/outputs/kabul_graph.* /content/drive/MyDrive/mphil-project/outputs/

In [13]:
!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the country containing Kandahar is named" \
  --target-token "Kabul" \
  --features '{"8": [2966, 1001], "16": [723, 1837, 392], "24": [88, 2529, 5009], "32": [6834]}'

Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 185.87it/s]

[1/3] Clean Model Baseline:
  - Top prediction: ' Kabul' (prob: 0.9883, logit: 21.88)
  - Target ' as': prob: 0.0011, logit: 15.06
  - Target 'K': prob: 0.0000, logit: 7.91
  - Target ' K': prob: 0.0059, logit: 16.75
  - Target ' Kabul': prob: 0.9883, logit: 21.88

[2/3] Running SAE Reconstruction-only Baseline (no features inhibited)...
Running forward pass with inhibition: {}...
  - Top prediction: ' Kabul' (prob: 0.9922, logit: 21.50)
  - Target ' as': prob: 0.0012, logit: 14.81
  - Target 'K': prob: 0.0000, logit: 7.03
  - Target ' K': prob: 0.0030, logit: 15.69
  - Target ' Kabul': prob: 0.9922, logit: 21.50

[3/3] Running Inhibition Intervention (inhibited features: {8: [2966, 1001], 16: [723, 1837, 392], 24: [88, 2529, 5009], 32: [6834]})...
Running forward pass with inhibition: {8: [2966,

In [ ]:
!python src/intervention.py \
  --mode swap \
  --prompt "Fact: The capital of the state containing Mesa is named" \
  --source-prompt "Fact: The capital of the country containing Kandahar is named" \
  --features '{"8": [2966, 1001], "16": [723, 1837, 392], "24": [88, 2529, 5009], "32": [6834]}' \
  --target-token "Kabul, Boston"

Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 187.10it/s]

[1/3] Clean Model Baseline:
  - Top prediction: ' after' (prob: 0.6328, logit: 18.00)
  - Target ' after': prob: 0.6328, logit: 18.00
  - Target ' Austin': prob: 0.0000, logit: 2.14
  - Target 'Austin': prob: 0.0000, logit: -5.91
  - Target 'Boston': prob: 0.0000, logit: 5.81
  - Target ' Boston': prob: 0.0055, logit: 13.25
  - Target ' Dallas': prob: 0.0000, logit: 0.95
  - Target ' what': prob: 0.0216, logit: 14.62
  - Target ' Worcester': prob: 0.2988, logit: 17.25

Source baseline prediction on 'Fact: The capital of the state containing Dallas is named':
  - Target ' after': prob: 0.1230, logit: 15.69
  - Target ' Austin': prob: 0.2773, logit: 16.50
  - Target 'Austin': prob: 0.0000, logit: 7.56
  - Target 'Boston': prob: 0.0000, logit: -3.36
  - Target ' Boston': prob: 0.0000, logit: 3.98
  

In [23]:
!cat /content/outputs/intervention_results.json

{
  "experiment_mode": "swap",
  "prompt": "Fact: the capital of the state containing Massachusetts is: \"",
  "clean_baseline": {
    "top_token": "The",
    "top_logit": 12.9375,
    "top_prob": 0.306640625,
    "targets": {
      "Austin": {
        "logit": 3.75,
        "prob": 3.123283386230469e-05
      }
    }
  },
  "source_prompt": "Fact: the capital of the state containing Dallas is: \"",
  "source_baseline": {
    "top_token": "The",
    "top_logit": 13.5,
    "top_prob": 0.28515625,
    "targets": {
      "Austin": {
        "logit": 13.125,
        "prob": 0.1962890625
      }
    }
  },
  "intervention": {
    "top_token": "The",
    "top_logit": 12.3125,
    "top_prob": 0.2119140625,
    "targets": {
      "Austin": {
        "logit": 3.0,
        "prob": 1.9073486328125e-05
      }
    }
  }
}

## Preventing Disconnects / Inactivity Timeouts

Because you have **Colab Pro+**, background execution is fully supported. You don't have to keep the tab open for the execution to finish as long as you've enabled background runtime settings.

However, if you want a classic browser-side keep-alive backup, you can paste the following JavaScript into your Web Browser Console (press `F12` or `Ctrl+Shift+I` > click **Console**):

```javascript
function ClickConnect(){
  console.log("Clicking Connect/Keep-alive...");
  document.querySelector("colab-connect-button").shadowRoot.querySelector("#connect").click()
}
setInterval(ClickConnect, 60000)
```
*(This clicks the Colab connection button every 60 seconds to emulate user activity.)*